In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install cdsapi xarray netCDF4 rioxarray rasterio

In [ ]:
import os

BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets"

folders = [

    "california/raw/era5/wind",
    "california/raw/era5/temperature",
    "california/raw/era5/humidity",

    "california/processed/wind",
    "california/processed/temperature",
    "california/processed/humidity",

    "california/grids/32x32",
    "california/grids/64x64",
]

for folder in folders:
    os.makedirs(f"{BASE}/{folder}", exist_ok=True)

print("California ERA5 folders created successfully.")

In [ ]:
%%writefile /root/.cdsapirc
url: https://cds.climate.copernicus.eu/api
key: e9b7a0af-311a-4edf-a94e-c580455fe680

In [ ]:
import cdsapi
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
CALIFORNIA_BBOX = {
    "north": 41.5,
    "south": 36.5,
    "west": -124.5,
    "east": -119.0
}

In [ ]:
import cdsapi

dataset = "reanalysis-era5-single-levels"

request = {
    "product_type": ["reanalysis"],

    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_dewpoint_temperature",
        "2m_temperature"
    ],

    "year": ["2025"],

    "month": ["06"],

    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30"
    ],

    "time": [
        "00:00", "06:00",
        "12:00", "18:00"
    ],

    "data_format": "netcdf",

    "download_format": "unarchived",

    "area": [
        CALIFORNIA_BBOX["north"],
        CALIFORNIA_BBOX["west"],
        CALIFORNIA_BBOX["south"],
        CALIFORNIA_BBOX["east"]
    ]
}

client = cdsapi.Client()

client.retrieve(
    dataset,
    request,
    "download.nc"
)

print("California ERA5 download complete.")

In [ ]:
import shutil

SOURCE_FILE = "download.nc"

DEST_FILE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/raw/era5/era5_california_june_2025.nc"

shutil.move(SOURCE_FILE, DEST_FILE)

print("California ERA5 file moved successfully.")

In [ ]:
import xarray as xr

ds = xr.open_dataset(DEST_FILE)

ds

In [ ]:
print(ds.data_vars)

In [ ]:
print(ds.dims)

In [ ]:
import matplotlib.pyplot as plt

temp = ds['t2m'].isel(valid_time=0)

plt.figure(figsize=(8,6))

temp.plot(cmap='hot')

plt.title("ERA5 California Temperature")

plt.show()

In [ ]:
u10 = ds['u10'].isel(valid_time=0)
v10 = ds['v10'].isel(valid_time=0)

plt.figure(figsize=(10,8))

plt.quiver(
    ds.longitude.values,
    ds.latitude.values,
    u10.values,
    v10.values
)

plt.title("California Wind Field")

plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()

In [ ]:
wind_x = ds['u10'].mean(dim='valid_time').values

wind_y = ds['v10'].mean(dim='valid_time').values

temperature = ds['t2m'].mean(dim='valid_time').values

dewpoint = ds['d2m'].mean(dim='valid_time').values

In [ ]:
temperature_c = temperature - 273.15

dewpoint_c = dewpoint - 273.15

In [ ]:
import numpy as np

humidity = 100 - 5 * (temperature_c - dewpoint_c)

humidity = np.clip(humidity, 0, 100)

In [ ]:
print("wind_x:", wind_x.shape)
print("wind_y:", wind_y.shape)
print("temperature:", temperature_c.shape)
print("humidity:", humidity.shape)

In [ ]:
def normalize(x):
    return (x - x.min()) / (x.max() - x.min())

In [ ]:
wind_x_norm = normalize(wind_x)

wind_y_norm = normalize(wind_y)

temp_norm = normalize(temperature_c)

humidity_norm = normalize(humidity)

In [ ]:
!pip install scikit-image

In [ ]:
from skimage.transform import resize

In [ ]:
GRID_32 = (32,32)
GRID_64 = (64,64)

wind_x_32 = resize(wind_x_norm, GRID_32)

wind_y_32 = resize(wind_y_norm, GRID_32)

temp_32 = resize(temp_norm, GRID_32)

humidity_32 = resize(humidity_norm, GRID_32)

wind_x_64 = resize(wind_x_norm, GRID_64)

wind_y_64 = resize(wind_y_norm, GRID_64)

temp_64 = resize(temp_norm, GRID_64)

humidity_64 = resize(humidity_norm, GRID_64)

In [ ]:
print(wind_x_32.shape)
print(temp_32.shape)

print(wind_x_64.shape)
print(temp_64.shape)

In [ ]:
import os

BASE = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids"

os.makedirs(f"{BASE}/32x32", exist_ok=True)

os.makedirs(f"{BASE}/64x64", exist_ok=True)

In [ ]:
np.save(f"{BASE}/32x32/wind_x.npy", wind_x_32)

np.save(f"{BASE}/32x32/wind_y.npy", wind_y_32)

np.save(f"{BASE}/32x32/temperature.npy", temp_32)

np.save(f"{BASE}/32x32/humidity.npy", humidity_32)

print("California 32x32 ERA5 tensors saved.")

In [ ]:
np.save(f"{BASE}/64x64/wind_x.npy", wind_x_64)

np.save(f"{BASE}/64x64/wind_y.npy", wind_y_64)

np.save(f"{BASE}/64x64/temperature.npy", temp_64)

np.save(f"{BASE}/64x64/humidity.npy", humidity_64)

print("California 64x64 ERA5 tensors saved.")

In [ ]:
plt.figure(figsize=(6,6))

plt.imshow(temp_32)

plt.title("California 32x32 Temperature Grid")

plt.colorbar()

plt.show()

In [ ]:
import os

grid_path = "/content/drive/MyDrive/PyroRL_Saudi_Project/datasets/california/grids/32x32"

print(os.listdir(grid_path))

In [ ]:
import numpy as np

wind_x = np.load(f"{grid_path}/wind_x.npy")

print(wind_x.shape)